In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
pip install transformers datasets sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [13]:
from datasets import load_dataset

dataset = load_dataset("thainq107/iwslt2015-en-vi")

train_ds = dataset["train"].select(range(100000))
val_ds = dataset["validation"].select(range(1268)) 

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name) 

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
max_length = 64

def preprocess(example):
    model_inputs = tokenizer(
        example["en"],
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["vi"],
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    # 🔥 bỏ padding khỏi loss
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs 

In [16]:
train_ds = train_ds.map(preprocess, batched=True, remove_columns=["en", "vi"])
val_ds = val_ds.map(preprocess, batched=True, remove_columns=["en", "vi"]) 

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./envi_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,        # 🔥 quan trọng
    num_train_epochs=3,
    logging_steps=200,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,                 # nếu dùng T4
    remove_unused_columns=False,
    report_to="none"
) 

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
) 

In [19]:
trainer.train() 

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,8.258911
400,6.359208
600,5.681594
800,5.274607
1000,5.057333
1200,4.829442
1400,4.682283
1600,4.513094
1800,4.425464
2000,4.347413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=37500, training_loss=2.850456785888672, metrics={'train_runtime': 7251.2715, 'train_samples_per_second': 41.372, 'train_steps_per_second': 5.172, 'total_flos': 5084754739200000.0, 'train_loss': 2.850456785888672, 'epoch': 3.0})

In [20]:
sentences = [
    "Hello!",
    "How are you?",
    "I am a student.",
    "This is a book.",
    "Nice to meet you."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=2
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50) 

EN: Hello!
VI: Xin chào !
--------------------------------------------------
EN: How are you?
VI: Các bạn thế nào ?
--------------------------------------------------
EN: I am a student.
VI: Tôi là một sinh viên .
--------------------------------------------------
EN: This is a book.
VI: Đây là một cuốn sách .
--------------------------------------------------
EN: Nice to meet you.
VI: Rất nhiều khi gặp bạn .
--------------------------------------------------


In [21]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

text = "Machine learning is very interesting."

inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_length=60,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True)) 

Các chiếc máy học rất thú vị .


In [22]:
sentences = [
    "Although it was raining, we decided to go out.",
    "If I had known earlier, I would have told you.",
    "The book that you gave me yesterday is very interesting.",
    "Even though he was tired, he continued working.",
    "She said that she would come if she had time."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=2
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50) 



EN: Although it was raining, we decided to go out.
VI: Nhưng nó đang trời , chúng tôi quyết định đi ra ngoài
--------------------------------------------------
EN: If I had known earlier, I would have told you.
VI: Nếu tôi biết được trước đó , tình sẽ nói với bạn .
--------------------------------------------------
EN: The book that you gave me yesterday is very interesting.
VI: Sách mà bạn cho tôi ngày hôm qua rất thú vị .
--------------------------------------------------
EN: Even though he was tired, he continued working.
VI: Mặc dù anh ấy rất dài , an hãy tiếp tục làm việc .
--------------------------------------------------
EN: She said that she would come if she had time.
VI: Cô ấy nói rằng cô sẽ đến nếu có thời gian .
--------------------------------------------------


In [23]:
sentences = [
    "The meeting has been postponed until next week.",
    "Please make sure to submit your assignment on time.",
    "This application helps users translate text quickly.",
    "We are developing a machine translation system.",
    "The results of this experiment are very promising."
]



for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=2
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50)

EN: The meeting has been postponed until next week.
VI: Các cuộc gặp đã được phụp cho đến tuần sau .
--------------------------------------------------
EN: Please make sure to submit your assignment on time.
VI: Hãy chắc rằng bạn làm việc của mình vào thời gian .
--------------------------------------------------
EN: This application helps users translate text quickly.
VI: Được thông dụng này giúp người dịch hơn .
--------------------------------------------------
EN: We are developing a machine translation system.
VI: Chúng tôi đang phát triển một hệ thống bản .
--------------------------------------------------
EN: The results of this experiment are very promising.
VI: Kết quả của cuộc thử nghiệm này rất mới .
--------------------------------------------------


In [24]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [25]:
import evaluate

bleu = evaluate.load("bleu")

In [41]:
raw_val = dataset["validation"].select(range(1000))

predictions = []
references = []

model.eval()

for example in raw_val:
    input_text = example["en"]

    # tokenize
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True).to(model.device)

    # generate
    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    # decode
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ref = example["vi"]

    predictions.append(pred)
    references.append([ref])  # 🔥 phải là list of list

# tính BLEU
result = bleu.compute(predictions=predictions, references=references)

print("BLEU:", result["bleu"] * 100)

BLEU: 14.14301756944678


In [42]:
model.save_pretrained("/kaggle/working/LLM-model")
tokenizer.save_pretrained("/kaggle/working/LLM-model")  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/LLM-model/tokenizer_config.json',
 '/kaggle/working/LLM-model/vocab.json',
 '/kaggle/working/LLM-model/source.spm',
 '/kaggle/working/LLM-model/target.spm',
 '/kaggle/working/LLM-model/added_tokens.json')

In [45]:
!zip -r LLM-model.zip /kaggle/working/LLM-model   

  adding: kaggle/working/LLM-model/ (stored 0%)
  adding: kaggle/working/LLM-model/config.json (deflated 64%)
  adding: kaggle/working/LLM-model/generation_config.json (deflated 43%)
  adding: kaggle/working/LLM-model/target.spm (deflated 50%)
  adding: kaggle/working/LLM-model/source.spm (deflated 51%)
  adding: kaggle/working/LLM-model/vocab.json (deflated 70%)
  adding: kaggle/working/LLM-model/model.safetensors (deflated 7%)
  adding: kaggle/working/LLM-model/tokenizer_config.json (deflated 67%)
